In [9]:
# =========================
# LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries loaded")

# =========================
# LOAD CLEAN TASK 1 DATA
# =========================
df = pd.read_csv("data/processed/clean_reviews.csv")

print("Data loaded:", df.shape)
print(df["bank"].value_counts())

# =========================
# SENTIMENT MODEL
# =========================
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# =========================
# SENTIMENT ANALYSIS
# =========================
texts = df["review"].astype(str).tolist()
results = sentiment_model(texts)

df["sentiment_label"] = [r["label"].lower() for r in results]
df["sentiment_score"] = [r["score"] for r in results]

# =========================
# TEXT CLEANING
# =========================
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df["clean"] = df["review"].apply(clean)

# =========================
# TF-IDF KEYWORDS
# =========================
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=1000)

X = vectorizer.fit_transform(df["clean"])
terms = vectorizer.get_feature_names_out()

keywords_by_bank = {}

for bank in df["bank"].unique():
    sub = df[df["bank"] == bank]
    vec = vectorizer.transform(sub["clean"])

    scores = np.asarray(vec.sum(axis=0)).flatten()
    top_idx = scores.argsort()[-10:][::-1]

    keywords_by_bank[bank] = [terms[i] for i in top_idx]

print("\nTop keywords per bank:")
for k, v in keywords_by_bank.items():
    print(k, ":", v)

# =========================
# THEME CLASSIFICATION
# =========================
def assign_theme(text):
    text = str(text).lower()

    if any(x in text for x in ["login", "otp", "password"]):
        return "Account Access Issues"

    if any(x in text for x in ["transfer", "transaction", "slow", "delay"]):
        return "Transaction Performance"

    if any(x in text for x in ["ui", "design", "interface"]):
        return "UI & Experience"

    if any(x in text for x in ["error", "bug", "crash"]):
        return "App Stability"

    if any(x in text for x in ["feature", "request", "add"]):
        return "Feature Requests"

    return "Other"

df["identified_theme"] = df["clean"].apply(assign_theme)

# =========================
# ANALYSIS
# =========================
print("\nSentiment by Bank:")
print(df.groupby("bank")["sentiment_score"].mean())

print("\nSentiment by Rating:")
print(df.groupby("rating")["sentiment_score"].mean())

print("\nTheme Distribution:")
print(df.groupby(["bank", "identified_theme"]).size())

# =========================
# FINAL OUTPUT (CORRECT FORMAT)
# =========================
df["review_id"] = df.index.astype(str)

final_df = df[[
    "review_id",
    "review",
    "sentiment_label",
    "sentiment_score",
    "identified_theme"
]]

final_df.to_csv("data/processed/task2_final_output.csv", index=False)

print("\nSaved ✔ task2_final_output.csv")
print("TASK 2 COMPLETE ✔")

Libraries loaded
Data loaded: (1500, 5)
bank
Commercial Bank of Ethiopia (CBE)    500
Dashen Bank                          500
Bank of Abyssinia (BOA)              500
Name: count, dtype: int64


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Top keywords per bank:
Commercial Bank of Ethiopia (CBE) : ['good', 'nice', 'app', 'best', 'ok', 'cbe', 'like', 'excellent', 'nice app', 'working']
Dashen Bank : ['good', 'app', 'nice', 'best', 'bank', 'dashen', 'great', 'super', 'easy', 'use']
Bank of Abyssinia (BOA) : ['good', 'app', 'best', 'nice', 'boa', 'working', 'bank', 'work', 'good app', 'mobile']

Sentiment by Bank:
bank
Bank of Abyssinia (BOA)              0.966775
Commercial Bank of Ethiopia (CBE)    0.976784
Dashen Bank                          0.976074
Name: sentiment_score, dtype: float64

Sentiment by Rating:
rating
1    0.979106
2    0.965342
3    0.977547
4    0.962809
5    0.972455
Name: sentiment_score, dtype: float64

Theme Distribution:
bank                               identified_theme       
Bank of Abyssinia (BOA)            Account Access Issues       15
                                   App Stability               15
                                   Feature Requests             4
                        